# P2-A1 — Your First LLM Calls + Prompting Basics

Welcome to Phase 2. You're done analysing data *about* models — now you **call** one. This task: make real LLM API calls (OpenAI) and learn the prompting levers that change output quality.

> Note: we're using OpenAI here because that's the key you have. Every concept (specificity, system prompts, few-shot) is provider-agnostic and transfers directly to Claude / LangGraph later.

## Setup (do this first)
1. Get an OpenAI API key → https://platform.openai.com/api-keys . Pay-as-you-go; we use the cheap **`gpt-4o-mini`**, so this whole task costs a fraction of a cent.
2. Create a file named `.env` **in the repo root** with one line:
   ```
   OPENAI_API_KEY=sk-...
   ```
   (`.env` is already git-ignored — your key never gets committed.)
3. Run the setup cell below. If it prints a greeting, you're wired up.

Model we'll use: `gpt-4o-mini` (fast + cheap, perfect for learning).

In [1]:
# --- Setup: this part is plumbing (given) ---
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                      # reads OPENAI_API_KEY from your .env
client = OpenAI()                  # picks up the key automatically

MODEL = 'gpt-4o-mini'

def ask(prompt, system=None, **kwargs):
    """Tiny helper: send a prompt, return the model's text reply."""
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **kwargs,
    )
    return resp.choices[0].message.content

# Smoke test
print(ask('In one sentence, say hello and confirm the API works.'))

Hello! The API is functioning properly.


## Task 1 — A vague prompt vs. a specific one
Pick a task (e.g. "write a product description for a water bottle"). Call `ask()` twice:
- once with a **vague** one-liner,
- once with a **specific** prompt (audience, tone, length, format constraints).

Print both outputs and eyeball the difference.
<br>Goal: *feel* how much specificity changes quality. This is 80% of prompting.

In [2]:
# TODO: vague_prompt vs specific_prompt, print both responses
print(ask('Write a description about a water bottle'))
print(ask('''**Title**  
Tips for Accelerating Gen AI Learning and Advancing to Senior AI Developer  

**Role & stance**  
You are a senior software engineer at Google, offering practical, experience‑based advice.  

**Task**  
Provide exactly five actionable tips that (a) help the user learn Generative AI more quickly and (b) guide the user toward becoming a senior AI developer.  

**Context**  
The user seeks concise guidance on fast‑tracking their Gen AI knowledge and career growth. No additional background is required.  

**Inputs available**  
- None (the prompt contains all necessary information).  

**Output requirements**  
- Format: numbered list (1‑5).  
- Each tip should be a single sentence or short paragraph, clearly addressing both learning speed and career progression.  
- Tone: professional, encouraging, and straightforward.  

**Constraints / Do-nots**  
- Do not exceed five tips.  
- Do not include unrelated advice (e.g., unrelated programming languages).  
- Do not use vague statements; each tip must be specific and actionable.  

**Examples / References**  
*(No examples provided; follow the format above.)*  

**Execution checklist**  
- [ ] Exactly five numbered tips are present.  
- [ ] Each tip is concise, actionable, and relevant to both learning Gen AI faster and becoming a senior AI developer.  
- [ ] No extraneous content or unrelated advice is included.  

**Conflict resolution**  
*(No conflicts identified.)*'''))

The EcoFlask water bottle is designed with both functionality and sustainability in mind. Made from high-grade stainless steel, it boasts a double-walled insulation to keep beverages hot for up to 12 hours and cold for up to 24 hours. The sleek, matte finish not only adds a modern touch but also ensures a comfortable grip, while the wide mouth opening makes it easy to fill with ice cubes or clean after use.

With a capacity of 32 ounces, the EcoFlask is perfect for hydration on the go, whether you're hitting the gym, embarking on a hike, or commuting to work. The leak-proof lid features a secure locking mechanism, making it safe to toss into any bag without the worry of spills. Best of all, this reusable water bottle is crafted from BPA-free materials, promoting an eco-friendly lifestyle and reducing single-use plastic waste.

Available in a variety of vibrant colors and patterns, the EcoFlask caters to personal style while encouraging healthy hydration habits. Whether you're an outdoo

## Task 2 — System prompt = persona/role
Send the **same** user question twice but with two different `system=` prompts (e.g. "You are a terse senior engineer" vs "You are an enthusiastic teacher for 5-year-olds"). Compare.
<br>Goal: see that the system prompt steers *behaviour/voice*, separate from the question itself.

In [4]:
# TODO: same question, two different system prompts
print(ask('Write a description about a water bottle', system='You are a professional product copywriter.'))
print(ask('Write a description about a water bottle', system='You are a technical writer for a scientific journal.'))

**SleekHydrate™ Insulated Water Bottle**

Stay refreshed on-the-go with the SleekHydrate™ Insulated Water Bottle, your ultimate hydration companion. Engineered with double-wall vacuum insulation, this bottle keeps your drinks icy cold for up to 24 hours and hot for up to 12, ensuring you enjoy the perfect temperature, whether you're at the gym, hiking in the mountains, or commuting to work.

Crafted from premium, food-grade stainless steel, the SleekHydrate™ is not only durable but also resistant to rust and punctures, making it ideal for any adventure. The BPA-free lid features a wide mouth opening for easy filling and cleaning, while the leak-proof design ensures you can toss it in your bag without worrying about spills.

With a range of vibrant colors and a sleek, ergonomic design, this water bottle easily fits into your lifestyle—whether you prefer working out, traveling, or just enjoying a day in the sun. Plus, it’s lightweight and portable, so you'll never be caught without your 

## Task 3 — Few-shot prompting (teach by example)
Make the model classify the sentiment of a sentence as exactly `positive` / `negative` / `neutral`. First try with **no examples**, then with **2–3 examples** baked into the prompt. Notice how examples lock the output format.
<br>Goal: few-shot examples are how you get consistent, parseable output without fine-tuning.

In [6]:
# zero-shot vs few-shot sentiment classification
review = 'The product was great, but the shipping was slow.'

# Zero-shot: no examples, no format lock — model rambles / hedges
print(ask(f'Classify the sentiment of this review: "{review}"'))

# Few-shot: 3 labeled examples teach the exact output format, then ask for the 4th
print(ask(f'''Classify the sentiment of each review as exactly one of: positive / negative / neutral.

Review: "Loved it, works perfectly and arrived early!"
Sentiment: positive
Review: "Broke after one day, total waste of money."
Sentiment: negative
Review: "It's okay, nothing special either way."
Sentiment: neutral
Review: "{review}"
Sentiment:'''))


The sentiment of this review can be classified as mixed. It expresses a positive sentiment regarding the product ("the product was great") but a negative sentiment concerning the shipping ("the shipping was slow").
neutral


## Task 4 — Explain (in your own words)
1. Of the three levers (specificity, system prompt, few-shot examples), which changed output quality the most for your tasks, and why?
2. You'll soon build RAG and agents on top of this. Why does *reliable, consistent* output formatting (Task 3) matter when an LLM's response feeds into the next step of a program?

> **1.** Specificity moved the needle most. In Task 1 the vague prompt ("write a description about a water bottle") returned a generic marketing blurb — fine, but I had no control over it. The specific prompt pinned down the role (senior engineer at Google), the exact count (5 tips), the format (numbered list), the tone, and explicit do-nots, and the output matched all of it. The system prompt (Task 2) clearly *steered the voice* — copywriter vs. technical-journal writer produced noticeably different structure and register — but it didn't change correctness or completeness the way specificity did; it shaped *how* it said things, not *what* it had to deliver. Few-shot would matter most when I need a rigid, repeatable output shape, but for these open-ended generation tasks the constraints I wrote into the prompt did the heavy lifting.
>
> **2.** Because in a program the model's output isn't the final answer — it's an *input to the next step*. In Task 3 the zero-shot reply was a friendly paragraph ("the sentiment is mixed...") that no downstream code could reliably parse, while the constrained prompt forced a single clean label (`Neutral`) I can read with one line of code. Once an LLM feeds a router, a retrieval step, a tool call, or a `json.loads()`, anything unstructured or unpredictable breaks the pipeline — the program can't branch on prose. Consistent formatting is what turns a chatty model into a dependable *component*: you can validate it, route on it, and chain it without the whole agent falling over on the next call.